In [1]:
!pip freeze

absl-py==2.1.0
anyio @ file:///croot/anyio_1729121277521/work
argon2-cffi @ file:///home/builder/ci_310/argon2-cffi_1641902187184/work
arrow==1.3.0
asttokens @ file:///home/conda/feedstock_root/build_artifacts/asttokens_1698341106958/work
async-lru @ file:///croot/async-lru_1699554519285/work
attrs @ file:///croot/attrs_1729089401488/work
audioread==3.0.1
Babel @ file:///croot/babel_1671781930836/work
beautifulsoup4 @ file:///croot/beautifulsoup4-split_1718029820055/work
bleach @ file:///opt/conda/conda-bld/bleach_1641577558959/work
bokeh @ file:///croot/bokeh_1727914487135/work
Bottleneck @ file:///croot/bottleneck_1731058641041/work
Brotli @ file:///croot/brotli-split_1714483155106/work
certifi @ file:///croot/certifi_1725551672989/work/certifi
cffi @ file:///croot/cffi_1726856441404/work
charset-normalizer @ file:///croot/charset-normalizer_1721748349566/work
click @ file:///croot/click_1698129812380/work
cloudpickle @ file:///croot/cloudpickle_1721657346512/work
comm @ file:///home

In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

from vocal_assistant.vocal_assistant import VocalAssistant
from vocal_assistant.emotion.emotion_model import process_func, EmotionModel
from transformers import Wav2Vec2Processor
from vocal_assistant.emotion.predict_emotion import load_trained_model, predict_emotion
import pandas as pd


/home/lucab/.conda/envs/recenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
songs = pd.read_pickle("./data/Songs")


In [3]:
vc = VocalAssistant(1)

Detected SSH environment; skipping pyttsx3 initialization.


In [4]:
device = 'cpu'

audeering_model_name = 'audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim'
audeering_processor = Wav2Vec2Processor.from_pretrained(audeering_model_name)
audeering_model = EmotionModel.from_pretrained(audeering_model_name).to(device)

custom_model_name = "custom_model.pth"
pretrained_model = "facebook/wav2vec2-base"
custom_model, custom_processor = load_trained_model(device, custom_model_name, pretrained_model)

/home/lucab/Recommersion/vocal_assistant/emotion/predict_emotion.py:489: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location=

Loaded trained model from checkpoint.


In [5]:
vc.talk("What is your mood today?")
"""while True:
    command, vocal_file = vc.take_command()
    print(command)
    break
""" 
file_path = "neutral.wav" 
vocal_file = vc.process_audio_file_str(file_path)
print("Audeering: ")
audeering = process_func(vocal_file, 16000)[0][:2]
print(audeering)
print("Custom: ")
custom = predict_emotion(custom_model, device, custom_processor, vocal_file)[0].tolist()
print(custom)
#custom model seems to give the same results: overfitting?
#[0.4807744026184082, 0.5821987390518188, 0.6608558893203735]


Simulating speech: What is your mood today?
Processing the file: neutral.wav


Numpy array shape: (71284,)
Recognized speech: recommend me something
Audeering: 
[0.26769602 0.39730906]
Custom: 
(3, 1, 40, 313)
[0.12285812199115753, 0.600323498249054]


In [13]:
import numpy as np

def playlist(dim_vec, songs, cut = 5):
    songs_list = pd.DataFrame({"id": songs["musicId"], "eucl_dist":songs[["Valence", "Arousal"]]\
                           .apply(lambda x: np.linalg.norm(x - dim_vec), axis=1), "Valence": songs["Valence"], "Arousal": songs["Arousal"],\
                            "title":songs["title"], "artist": songs["artist"], "mp3_file":songs["mp3_file"]})

    return songs_list.sort_values(by="eucl_dist")[:cut]

aud_dim_vec = np.array(audeering[0:2])
custom_dim_vec = np.array(custom)

aud_songs_list = playlist(aud_dim_vec, songs)
custom_songs_list = playlist(custom_dim_vec, songs)

print(aud_songs_list)
print(custom_songs_list)
#create temp dir?

       id  eucl_dist   Valence   Arousal  \
34   3035   0.002881  0.266667  0.400000   
585   750   0.005851  0.262500  0.400000   
121   149   0.011100  0.262500  0.387500   
71   1095   0.012435  0.255556  0.400000   
599  1776   0.013135  0.277778  0.388889   

                                         title           artist  \
34   Mit dem Unibalett im Ulmer Monster tanzen  Die Partysahnen   
585                                   Magnolia    PLAYBOI CARTI   
121                                  Fire Away  Chris Stapleton   
71                                 Old Strange       Steve Gunn   
599      Through The Haze Of The Green Glasses     Ghost Hunter   

                                              mp3_file  
34   [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  
585  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  
121  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  
71   [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  
599  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0

In [18]:
import sounddevice as sd

for i in range(len(aud_songs_list)):
    sd.play(aud_songs_list["mp3_file"].iloc[i], 44100)
    sd.wait()

for i in range(len(custom_songs_list)):
    sd.play(custom_songs_list["mp3_file"].iloc[i], 44100)
    sd.wait()


Mit dem Unibalett im Ulmer Monster tanzen
repr
Magnolia
repr
Fire Away
repr
Old Strange
repr
Through The Haze Of The Green Glasses
repr


KeyboardInterrupt: 

In [ ]:
pd.read_pickle("./data/IEMOCAP_useful")

ModuleNotFoundError: No module named 'numpy._core.numeric'